# Exploring the chunker - interactive playground

The chunker turns raw source files into `CodeChunk` objects for the RAG ingestion
pipeline: tree-sitter parses code into a syntax tree, we walk that tree, and every
function / class / method node is cut out as a chunk.

**How to use:** run the cells top to bottom. Kernel: the project `venv`
(in VS Code: Select Kernel -> Python Environments -> venv, Python 3.13).

| Cell | Section | What you will see |
|---|---|---|
| 1 | Setup | import the shared pieces from app/ingestion/chunker.py |
| 2 | STEP 0 | create example.py (the file your original snippet was missing) |
| 3 | Look at raw tree | the parse tree - your original exploration snippet |
| 4 | Poke one node | inspect a single function_definition node |
| 5 | STEP 1 | collect imports for the dependency graph |
| 6 | STEP 2 | chunk_file(): tree -> list[CodeChunk] |
| 7 | STEP 3 | chunk_repository(): chunk whole folders |
| 8 | STEP 4 | mini test suite as quick asserts |

Keep this notebook in the project root so the relative paths work.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()  # notebook lives in the project root
sys.path.insert(0, str(PROJECT_ROOT))

# shared building blocks still live in app/ingestion/chunker.py
from app.ingestion.chunker import (
    CodeChunk,
    _get_parser_and_lang,
    FUNCTION_NODE_TYPES,
    IMPORT_NODE_TYPES,
)

print("chunker module imported")
print("FUNCTION_NODE_TYPES =", FUNCTION_NODE_TYPES)

chunker module imported
FUNCTION_NODE_TYPES = {'class_definition', 'constructor_declaration', 'function_declaration', 'function_definition', 'method_declaration', 'method_definition'}


## STEP 0 - a sample file to chew on

Your original snippet read `example.py`, which never existed (that is why
`chunker.py` crashed on import before). This cell creates it - edit the code
below and re-run as much as you like.

In [2]:
EXAMPLE = '''import os
from pathlib import Path


def top_level_function(a, b):
    # adds two numbers
    return a + b


class Calculator:
    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b
'''

Path("example.py").write_text(EXAMPLE, encoding="utf-8")
print(Path("example.py").read_text())

import os
from pathlib import Path


def top_level_function(a, b):
    # adds two numbers
    return a + b


class Calculator:
    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b



In [3]:
# look at raw tree - your original exploration snippet (example.py now exists!)
source = Path("example.py").read_bytes()
parser, lang = _get_parser_and_lang("example.py")
tree = parser.parse(source)

def walk(node, depth=0, max_depth=None):
    # L{line}:{col} - tree-sitter rows are 0-based, +1 makes them match the editor
    start = f"L{node.start_point.row + 1}:{node.start_point.column}"
    end = f"L{node.end_point.row + 1}:{node.end_point.column}"
    print("  " * depth + f"{node.type} [{start} - {end}]")
    if max_depth is not None and depth >= max_depth:
        return
    for child in node.children:
        walk(child, depth + 1, max_depth)

walk(tree.root_node)                 # full tree
# walk(tree.root_node, max_depth=2)  # <- shallower view: uncomment to try

module [L1:0 - L16:0]
  import_statement [L1:0 - L1:9]
    import [L1:0 - L1:6]
    dotted_name [L1:7 - L1:9]
      identifier [L1:7 - L1:9]
  import_from_statement [L2:0 - L2:24]
    from [L2:0 - L2:4]
    dotted_name [L2:5 - L2:12]
      identifier [L2:5 - L2:12]
    import [L2:13 - L2:19]
    dotted_name [L2:20 - L2:24]
      identifier [L2:20 - L2:24]
  function_definition [L5:0 - L7:16]
    def [L5:0 - L5:3]
    identifier [L5:4 - L5:22]
    parameters [L5:22 - L5:28]
      ( [L5:22 - L5:23]
      identifier [L5:23 - L5:24]
      , [L5:24 - L5:25]
      identifier [L5:26 - L5:27]
      ) [L5:27 - L5:28]
    : [L5:28 - L5:29]
    comment [L6:4 - L6:23]
    block [L7:4 - L7:16]
      return_statement [L7:4 - L7:16]
        return [L7:4 - L7:10]
        binary_operator [L7:11 - L7:16]
          identifier [L7:11 - L7:12]
          + [L7:13 - L7:14]
          identifier [L7:15 - L7:16]
  class_definition [L10:0 - L15:20]
    class [L10:0 - L10:5]
    identifier [L10:6 - L10:16]
    : 

In [4]:
# Notebooks shine here: grab ONE node and interrogate it.
fn_node = next(n for n in tree.root_node.children if n.type == "function_definition")

print("node type:  ", fn_node.type)
print("name field: ", fn_node.child_by_field_name("name").text.decode())
print("line span:  ", fn_node.start_point.row + 1, "->", fn_node.end_point.row + 1)
print("byte span:  ", fn_node.start_byte, "->", fn_node.end_byte)
print("text:")
print(fn_node.text.decode())

# more things to try:
# fn_node.children                  # direct children
# [c.type for c in fn_node.children]
# fn_node.parent                    # walk back up

node type:   function_definition
name field:  top_level_function
line span:   5 -> 7
byte span:   41 -> 112
text:
def top_level_function(a, b):
    # adds two numbers
    return a + b


## STEP 1 - collect imports (for the dependency graph)

In [5]:
def _node_text(source: bytes, node) -> str:
    # slice the raw bytes for a node (portable across tree-sitter versions)
    return source[node.start_byte:node.end_byte].decode("utf-8", errors="replace")

def _extract_imports(source: bytes, root_node, lang: str) -> list[str]:
    # collect every import statement in the file (order kept, duplicates removed)
    imports = []

    def visit(node):
        if node.type in IMPORT_NODE_TYPES[lang]:
            text = _node_text(source, node)
            if text and text not in imports:
                imports.append(text)
        for child in node.children:
            visit(child)

    visit(root_node)
    return imports


_extract_imports(source, tree.root_node, lang)

['import os', 'from pathlib import Path']

## STEP 2 - chunk_file(): parse tree -> list of CodeChunk

`Calculator` **and** its methods each become their own chunk. The overlap is
intentional: smaller chunks embed better for retrieval.

In [6]:
def chunk_file(file_path: str) -> list[CodeChunk]:
    # parse one file -> one CodeChunk per function / class / method
    parser, lang = _get_parser_and_lang(file_path)
    if parser is None:
        return []  # unsupported extension -> nothing to chunk

    source = Path(file_path).read_bytes()
    tree = parser.parse(source)
    imports = _extract_imports(source, tree.root_node, lang)

    chunks: list[CodeChunk] = []

    def visit(node):
        if node.type in FUNCTION_NODE_TYPES:
            name_node = node.child_by_field_name("name")
            name = _node_text(source, name_node) if name_node is not None else "<anonymous>"
            chunks.append(CodeChunk(
                file_path=str(file_path),
                function_name=name,
                start_line=node.start_point.row + 1,  # tree-sitter rows are 0-based
                end_line=node.end_point.row + 1,      # -> 1-based, inclusive
                content=_node_text(source, node),
                imports=list(imports),
            ))
        for child in node.children:
            visit(child)

    visit(tree.root_node)
    return chunks


chunks = chunk_file("example.py")
print(f"{len(chunks)} chunks from example.py")
print()
for c in chunks:
    print(f"--- {c.function_name}  (lines {c.start_line}-{c.end_line})")
    print(f"    imports: {c.imports}")
    print(c.content)

4 chunks from example.py

--- top_level_function  (lines 5-7)
    imports: ['import os', 'from pathlib import Path']
def top_level_function(a, b):
    # adds two numbers
    return a + b
--- Calculator  (lines 10-15)
    imports: ['import os', 'from pathlib import Path']
class Calculator:
    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b
--- add  (lines 11-12)
    imports: ['import os', 'from pathlib import Path']
def add(self, a, b):
        return a + b
--- multiply  (lines 14-15)
    imports: ['import os', 'from pathlib import Path']
def multiply(self, a, b):
        return a * b


## STEP 3 - chunk_repository(): chunk whole folders

This is what the ingestion pipeline will call after `clone.py` brings a repo
into `repos/`.

In [7]:
SKIP_DIRS = {".git", "__pycache__", "venv", ".venv", "node_modules", "build", "dist", ".idea", ".vscode"}

def chunk_repository(repo_path: str) -> list[CodeChunk]:
    # walk a folder and chunk every supported source file in it
    chunks = []
    for path in sorted(Path(repo_path).rglob("*")):
        if not path.is_file():
            continue
        if any(part in SKIP_DIRS for part in path.parts):
            continue
        chunks.extend(chunk_file(str(path)))
    return chunks


all_chunks = chunk_repository(str(PROJECT_ROOT / "app"))
print(f"{len(all_chunks)} chunks in app/")
print()
for c in all_chunks:
    print(f"  {c.function_name:<25} lines {c.start_line:>3}-{c.end_line:<3} {Path(c.file_path).name}")

19 chunks in app/

  LLMClient                 lines   5-72  llm_client.py
  __init__                  lines   6-13  llm_client.py
  _call_groq                lines  15-29  llm_client.py
  _call_openrouter          lines  31-45  llm_client.py
  _call_gemini              lines  47-55  llm_client.py
  chat                      lines  57-72  llm_client.py
  RateLimitError            lines  75-76  llm_client.py
  Settings                  lines   3-15  config.py
  CodeChunk                 lines  27-33  chunker.py
  _get_parser_and_lang      lines  35-46  chunker.py
  extract_imports           lines  48-59  chunker.py
  walk_node                 lines  52-57  chunker.py
  extract_chunks            lines  62-79  chunker.py
  walk_node                 lines  64-77  chunker.py
  chunk_file                lines  82-97  chunker.py
  chunk_repo                lines 100-112 chunker.py
  clean_repo                lines   8-32  clone.py
  get_repo_name             lines  34-35  clone.py
  main     

## STEP 4 - mini test suite (quick asserts)

`tests/test_chunker.py` was rolled back to empty together with the rest of the
changes - the same checks live here now. Re-run this cell after every tweak.

In [8]:
import tempfile

def _write(tmp, name, content):
    p = tmp / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return p

tmp = Path(tempfile.mkdtemp())

# 1) functions, classes and methods all become chunks
f = _write(tmp, "sample.py", EXAMPLE)
cs = chunk_file(str(f))
names = [c.function_name for c in cs]
assert {"top_level_function", "Calculator", "add", "multiply"} <= set(names), names
assert len(cs) == 4
print("OK 1 - functions / classes / methods chunked")

# 2) imports captured for the dependency graph
assert all("import os" in c.imports for c in cs)
assert all(any(i.startswith("from pathlib") for i in c.imports) for c in cs)
print("OK 2 - imports captured")

# 3) line numbers point at real code
fn = next(c for c in cs if c.function_name == "top_level_function")
lines = f.read_text(encoding="utf-8").splitlines()
assert lines[fn.start_line - 1].startswith("def top_level_function")
assert lines[fn.end_line - 1].strip() == "return a + b"
print("OK 3 - line ranges point at real code")

# 4) unsupported extensions are ignored
_write(tmp, "notes.md", "# just markdown")
assert chunk_file(str(tmp / "notes.md")) == []
print("OK 4 - .md ignored")

# 5) javascript works too
js = 'import { useState } from "react";\n\nfunction greet(name) {\n  return name;\n}\n\nclass Widget {\n  render() {\n    return 1;\n  }\n}\n'
_write(tmp, "widget.js", js)
cs_js = chunk_file(str(tmp / "widget.js"))
assert "greet" in [c.function_name for c in cs_js]
assert "render" in [c.function_name for c in cs_js]
print("OK 5 - javascript chunked")

# 6) repository walk skips noise dirs
repo = tmp / "repo"
_write(repo, "pkg/mod.py", "def real_function():\n    return 42\n")
_write(repo, "pkg/__pycache__/junk.py", "def junk():\n    return 0\n")
_write(repo, "pkg/readme.md", "docs only")
_write(repo, "pkg/venv/thing.py", "def in_venv():\n    return 1\n")
assert [c.function_name for c in chunk_repository(str(repo))] == ["real_function"]
print("OK 6 - repo walk skips __pycache__ / venv / .md")

print()
print("ALL CHECKS PASSED")

OK 1 - functions / classes / methods chunked
OK 2 - imports captured
OK 3 - line ranges point at real code
OK 4 - .md ignored
OK 5 - javascript chunked
OK 6 - repo walk skips __pycache__ / venv / .md

ALL CHECKS PASSED


## Next steps

Happy with the logic? Paste `chunk_file`, `chunk_repository` (and `SKIP_DIRS`)
back into `app/ingestion/chunker.py` - the module itself was rolled back to your
original version (languages, node types, `CodeChunk`, `_get_parser_and_lang`).

In [9]:
from pathlib import Path

source = Path("example.py").read_bytes()
parser, lang = _get_parser_and_lang("example.py")
tree = parser.parse(source)

def walk(node, depth=0):
    print("  " * depth + node.type)
    for child in node.children:
        walk(child, depth + 1)

walk(tree.root_node)

module
  import_statement
    import
    dotted_name
      identifier
  import_from_statement
    from
    dotted_name
      identifier
    import
    dotted_name
      identifier
  function_definition
    def
    identifier
    parameters
      (
      identifier
      ,
      identifier
      )
    :
    comment
    block
      return_statement
        return
        binary_operator
          identifier
          +
          identifier
  class_definition
    class
    identifier
    :
    block
      function_definition
        def
        identifier
        parameters
          (
          identifier
          ,
          identifier
          ,
          identifier
          )
        :
        block
          return_statement
            return
            binary_operator
              identifier
              +
              identifier
      function_definition
        def
        identifier
        parameters
          (
          identifier
          ,
          identifier
     

In [11]:
from app.ingestion.chunker import extract_chunks
test_source = b'''
class Foo:
    def bar(self):
        pass
    def baz(self):
        pass
'''
tree = parser.parse(test_source)
chunks = extract_chunks(tree.root_node, test_source, "test.py", [])
for c in chunks:
    print(c.function_name, c.start_line, c.end_line)

TypeError: 'int' object is not subscriptable